# CLOUD-04 — public CPU-only Colab pilot (draft)
**Do not run on GPU.** This is a separate manual public experiment, not production audiobook rendering. Do not mount Drive, upload the manuscript, voice, latents, credentials, or labels. Built-in Silero speaker and four invented public phrases are reused from the reviewed Kaggle smoke. No paid resources. Select a free CPU runtime; verify account limits and retain 20% reserve. If quota or cost cannot be confirmed, do not proceed. Change all four admission booleans only after verifying these conditions. No unattended run or automatic retry. The child processes have timeouts, but these do not prove provider shutdown. Save the verified public WAVs and receipt before disconnecting/deleting the runtime manually. Never count this pilot as proof of author-voice quality. Notebook draft, runtime and model compatibility not yet qualified.

In [ ]:
# Admission is manual; running all cells without explicit changes fails closed.
import os, sys, json, hashlib, pathlib, tempfile, time, subprocess
from datetime import datetime, timezone

APPROVE_CPU = False
CONFIRM_FREE_CPU_RUNTIME = False
CONFIRM_NO_DRIVE_OR_PRIVATE_INPUT = False
CONFIRM_QUOTA_AND_RESERVE = False
if not all((APPROVE_CPU, CONFIRM_FREE_CPU_RUNTIME,
            CONFIRM_NO_DRIVE_OR_PRIVATE_INPUT, CONFIRM_QUOTA_AND_RESERVE)):
    raise RuntimeError("MANUAL_CPU_ADMISSION_REQUIRED")
if "google.colab" not in sys.modules:
    import google.colab
if any(os.environ.get(k) for k in ("COLAB_GPU",) if os.environ.get(k) not in ("0", "", None)):
    raise RuntimeError("CPU_RUNTIME_REQUIRED")
if any(os.environ.get(k) for k in ("HF_TOKEN", "GITHUB_TOKEN", "KAGGLE_KEY", "KAGGLE_API_TOKEN")):
    raise RuntimeError("UNEXPECTED_CREDENTIAL_ENV")
if (pathlib.Path.cwd() / "drive").exists():
    raise RuntimeError("DETACH_DRIVE_FIRST")
import torch
if torch.cuda.is_available():
    raise RuntimeError("CPU_RUNTIME_REQUIRED")
ROOT = pathlib.Path(tempfile.mkdtemp(prefix="audiobook-public-cpu-"))
START = time.monotonic()
print("CPU admission recorded; public synthetic cases only.")

In [ ]:
# Explicit public download, separate from inference. No model is loaded here.
URL = "https://models.silero.ai/models/tts/ru/v5_cis_base.pt"
SHA = "ba41b18f6a707ad93605a162998865e7c087153d2e010a26dd02229dab0e672a"
download = r'''
import hashlib, pathlib, sys, urllib.request
url, expected, target = sys.argv[1:]
p = pathlib.Path(target)
h = hashlib.sha256()
size = 0
with urllib.request.urlopen(url, timeout=30) as response, p.open("xb") as f:
    if response.geturl() != url:
        raise RuntimeError("UNEXPECTED_REDIRECT")
    while chunk := response.read(1024 * 1024):
        size += len(chunk)
        if size > 1024**3:
            raise RuntimeError("DOWNLOAD_LIMIT")
        h.update(chunk)
        f.write(chunk)
if h.hexdigest() != expected:
    raise RuntimeError("MODEL_HASH_MISMATCH")
'''
MODEL = ROOT / "silero.pt"
remaining = 600 - (time.monotonic() - START)
if remaining <= 0:
    raise RuntimeError("ADMISSION_WINDOW_EXPIRED")
t0 = time.monotonic()
subprocess.run([sys.executable, "-c", download, URL, SHA, str(MODEL)],
               check=True, timeout=min(180, remaining), cwd=ROOT)
DOWNLOAD_S = time.monotonic() - t0
print("Pinned public asset verified:", SHA)

In [ ]:
# CPU-only public built-in speaker pilot. No reference voice or private text.
# Child process is bounded; a timeout is not provider-session termination.
pilot = r'''
import hashlib, json, pathlib, sys, time, wave
import torch
import numpy as np
root = pathlib.Path(sys.argv[1])
expected = sys.argv[2]
model_path = root / "silero.pt"
with model_path.open("rb") as f:
    if hashlib.file_digest(f, "sha256").hexdigest() != expected:
        raise RuntimeError("MODEL_HASH_MISMATCH")
if torch.cuda.is_available():
    raise RuntimeError("CPU_RUNTIME_REQUIRED")
torch.set_num_threads(2)
torch.manual_seed(42)
t0 = time.perf_counter()
model = torch.package.PackageImporter(str(model_path)).load_pickle("tts_models", "model")
model.to(torch.device("cpu"))
speaker = "ru_bel_dmitriy"
if speaker not in model.speakers:
    raise RuntimeError("SPEAKER_UNAVAILABLE")
setup_s = time.perf_counter() - t0
cases = (
    "На холм+е сто+ял з+амок.",
    "На двер+и вис+ел зам+ок.",
    "На стол+е леж+али хл+еб, с+ыр и мук+а.",
    "+Это был+а насто+ящая м+ука.",
)
rows = []
for i, case in enumerate(cases, 1):
    torch.manual_seed(42)
    start = time.perf_counter()
    with torch.no_grad():
        wav = model.apply_tts(text=case, speaker=speaker, sample_rate=24000)
    elapsed = time.perf_counter() - start
    audio = wav.detach().cpu().numpy()
    if audio.ndim != 1 or not 24000 <= len(audio) <= 30*24000 or not np.isfinite(audio).all() or not np.any(audio):
        raise RuntimeError("INVALID_AUDIO")
    name = f"clip-{i:02d}.wav"
    final = root / name
    tmp = root / (name + ".part")
    pcm = (np.clip(audio, -1, 1) * 32767).astype("<i2")
    with wave.open(str(tmp), "wb") as w:
        w.setnchannels(1)
        w.setsampwidth(2)
        w.setframerate(24000)
        w.writeframes(pcm.tobytes())
    tmp.replace(final)
    rows.append(dict(file=name, sha256=hashlib.sha256(final.read_bytes()).hexdigest(),
                     samples=len(audio), inference_s=elapsed, rtf=elapsed/(len(audio)/24000)))
receipt = dict(policy="colab-public-silero-cpu-v1", device="cpu", status="READY",
               model_sha256=expected, torch=str(torch.__version__), seed=42, threads=2,
               setup_s=setup_s, cases=rows, release_ready=False)
tmp = root / "receipt.json.part"
tmp.write_text(json.dumps(receipt, indent=2), encoding="utf-8")
tmp.replace(root / "receipt.json")
'''
if time.monotonic() - START >= 600:
    raise RuntimeError("ADMISSION_WINDOW_EXPIRED")
remaining = 600 - (time.monotonic() - START)
env = dict(os.environ, HF_HUB_OFFLINE="1", TRANSFORMERS_OFFLINE="1",
           CUDA_VISIBLE_DEVICES="")
subprocess.run([sys.executable, "-c", pilot, str(ROOT), SHA],
               check=True, timeout=min(360, remaining), cwd=ROOT, env=env)
receipt = json.loads((ROOT / "receipt.json").read_text(encoding="utf-8"))
for row in receipt["cases"]:
    if hashlib.sha256((ROOT / row["file"]).read_bytes()).hexdigest() != row["sha256"]:
        raise RuntimeError("OUTPUT_HASH_MISMATCH")
receipt.update(download_s=DOWNLOAD_S, elapsed_s=time.monotonic()-START,
               quota_evidence="manual_attestation_only",
               provider_shutdown_verified=False,
               finished_utc=datetime.now(timezone.utc).isoformat())
verified = ROOT / "receipt.verified.json"
tmp = ROOT / "receipt.verified.json.part"
tmp.write_text(json.dumps(receipt, indent=2), encoding="utf-8")
tmp.replace(verified)
print(json.dumps(receipt, ensure_ascii=False, indent=2))
print("Save only public WAVs and receipt via the Colab Files panel, then disconnect and delete runtime.")